# Ejemplo: electrostática

:::{seealso} Revisa la física
Puedes revisar la teoría en un libro de introducción al electromagnetismo
como @Purcell2013.
:::

> Escriba una función que calcule la magnitud y dirección en cualquier punto
> del campo eléctrico que generan
> 35 partículas cuyas posiciones y cargas obedecen las siguientes propiedades:
>
> $$ \mathbf{r}_{ij} = (7\ \text{m}) \left( \cos{\frac{2 \pi i}{5}} \hat{\mathbf{x}} + \sin{\frac{2 \pi i}{5}} \hat{\mathbf{y}} \right) + (j - 4) (1\ \text{m}) \hat{\mathbf{z}} $$
> con $ 1 \leq i \leq 5 $ y $ 1 \leq j \leq 7 $
> (_i.e._ siete aros de 5 partículas, uno sobre el otro) y
> $$
>   q_{ij} = (i - j) (10^{-6} \, \text{C}).
> $$

Sí, este es un ejercicio artificial, pero impráctico de resolver a mano. Por supuesto,
para una computadora no es nada.

El campo eléctrico debido a una distribución arbitraria de cargas puntuales es
$$
  \mathbf{E}(\mathbf{x}) = \frac{1}{4 \pi \epsilon_0}
  \sum_{j} q_j \frac{\mathbf{r} - \mathbf{r}_j}{{\lvert \mathbf{r} - \mathbf{r}_j \rvert}^3}
$$

In [ ]:
import math

En este ejercicio necesitamos vectores, ya que estaremos tratando con magnitudes
y direcciones. Ni Python base ni el módulo `math` proveen estructuras de vectores, por
lo que usaremos listas.

In [ ]:
# Python no sabe lo que es un vector.
print([1.0, 2.0, 3.0] + [2.0, 4.0, 5.0])
print("El operador `+` concatena listas. El operador `-` generaría un error.")

Crearemos una lista que contenga las coordenadas de la $j$-ésima partícula en su
$(j-1)$-ésima posición (recordemos que los índices en Python empiezan en 0).
Luego haremos lo mismo para las cargas. Las cargas son cantidades escalares, por lo que
tendremos un arreglo de `float`s.

In [ ]:
positions: list[list[float]] = [
    [
        7.0 * math.cos(2.0 * math.pi * i / 5),
        7.0 * math.sin(2.0 * math.pi * i / 5),
        (j - 4),
    ]
    for i in range(1, 6)
    for j in range(1, 8)
]
# nota que usamos posiciones en metros

charges: list[float] = [(i - j) * 1.0e-6 for i in range(1, 6) for j in range(1, 8)]

Ahora implementemos una función que toma una lista de coordenadas y una lista de cargas
y calcula el campo eléctrico en una posición dada.

En una aplicación tal vez sería más conveniente implementar partículas cargadas como
objetos, pero esto es demasiado complejo para este problema específico.

In [ ]:
def efield_points(
    r: list[float], rs: list[list[float]], qs: list[float]
) -> list[float]:
    """
    Dada una lista de posiciones rs [m] y de cargas qs [C] para cada posición, calcula el
    campo eléctrico [N/C] en r [m].
    """

    if len(rs) != len(qs):
        raise ValueError("rs y qs no son de la misma dimensión")

    if any([len(ri) != 3 for ri in rs]):
        raise TypeError("Todas las posiciones deben ser vectores 3D.")

    epsilon_0 = 8.854e-12  # C^2 / N / m^2

    E = [
        (1.0 / 4.0 / math.pi / epsilon_0)
        * sum(
            qs[i]
            * (r[k] - rs[i][k])
            / (sum((r[m] - rs[i][m]) ** 2.0 for m in range(3))) ** 1.5
            for i in range(len(rs))
        )
        for k in range(3)
    ]

    return E

Este es un código un poco difícil de leer. Como Python no incluye vectores, debemos ser
específicos con los índices de nuestras listas.

La función `efield_points` nos da un "vector" cartesiano para el campo eléctrico.
Tal vez sea más conveniente convertir a coordenadas esféricas.

In [ ]:
def spherical(x: float, y: float, z: float) -> list[float]:
    """
    Convierte coordenadas cartesianas a las coordenadas [r, theta, phi] correspondientes.
    """

    r = math.hypot(x, y, z)

    theta = math.acos(z / r)

    phi = math.atan2(y, x)

    return [r, theta, phi]

In [ ]:
E0 = efield_points([0.0, 0.0, 0.0], positions, charges)
r0, theta0, phi0 = spherical(E0[0], E0[1], E0[2])

print(
    f"En el origen el campo es E = {r0:.2e} N/C",
    f"con ángulo θ = {math.degrees(theta0):.1f}°",
    f"y φ = {math.degrees(phi0):.1f}°.",
)

In [ ]:
Efar = efield_points([5.0, 7.6, 5.67], positions, charges)
r_far, theta_far, phi_far = spherical(Efar[0], Efar[1], Efar[2])

print(
    f"En el punto lejano el campo es E = {r_far:.2e} N/C",
    f"con ángulo θ = {math.degrees(theta_far):.1f}°",
    f"y φ = {math.degrees(phi_far):.1f}°.",
)